# 10 — Final Model + Calibration + Threshold + Production Bundle

Purpose:
- load the selected candidate;
- refit using train + validation groups;
- generate grouped out-of-fold probabilities for calibration;
- tune the operating threshold without using the final test set;
- evaluate once on the untouched test users;
- generate KEEP / REVIEW / REMOVE recommendations;
- package the full inference artifact for Lambda/backend.

In [1]:
# AI-Based IAM Permission Optimizer -- Notebook 10: Final Model
from pathlib import Path
import json, platform, sys
import joblib, numpy as np, pandas as pd
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, balanced_accuracy_score,
    matthews_corrcoef, brier_score_loss, classification_report, confusion_matrix,
)

PROJECT_DIR  = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR     = PROJECT_DIR / "data"
ARTIFACT_DIR = PROJECT_DIR / "artifacts"
OUTPUT_DIR   = PROJECT_DIR / "outputs"

df     = pd.read_csv(DATA_DIR / "labeled_dataset.csv")
splits = pd.read_csv(DATA_DIR / "splits.csv")
df["split"] = splits["split"].values

for _sel_path in [OUTPUT_DIR / "model_selection.json", ARTIFACT_DIR / "model_selection.json"]:
    if _sel_path.exists():
        with open(_sel_path, encoding="utf-8") as f:
            selection = json.load(f)
        break
else:
    raise FileNotFoundError("model_selection.json not found")

selected_name        = selection.get("selected_model_name", selection.get("selected_model", "xgboost"))
selected_artifact    = joblib.load(ARTIFACT_DIR / f"{selected_name}_candidate.joblib")
feature_columns      = selected_artifact["feature_columns"]
numeric_features     = selected_artifact["numeric_features"]
categorical_features = selected_artifact["categorical_features"]
estimator            = selected_artifact["estimator"]

X      = df[feature_columns].copy()
y      = df["target"].astype(int)
groups = df["user_id"]

train_pool_mask   = df["split"].isin(["train", "validation"])
test_mask         = df["split"].eq("test")
X_train_pool      = X.loc[train_pool_mask].reset_index(drop=True)
y_train_pool      = y.loc[train_pool_mask].reset_index(drop=True)
groups_train_pool = groups.loc[train_pool_mask].reset_index(drop=True)
X_test            = X.loc[test_mask].reset_index(drop=True)
y_test            = y.loc[test_mask].reset_index(drop=True)

print("Selected model     :", selected_name)
print("Final training rows:", len(X_train_pool))
print("Final test rows    :", len(X_test))


Selected model     : xgboost
Final training rows: 16000
Final test rows    : 4000


In [2]:
# Grouped OOF raw probabilities on the complete training pool.
def grouped_oof_probabilities(estimator, X_data, y_data, groups_data, n_splits=5):
    cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=45)
    oof = np.full(len(X_data), np.nan)
    for fold, (fit_idx, valid_idx) in enumerate(cv.split(X_data, y_data, groups=groups_data), start=1):
        fold_model = clone(estimator)
        fold_model.fit(X_data.iloc[fit_idx], y_data.iloc[fit_idx])
        oof[valid_idx] = fold_model.predict_proba(X_data.iloc[valid_idx])[:, 1]
        print(f"OOF fold {fold} complete")
    if np.isnan(oof).any():
        raise RuntimeError("Missing OOF probabilities.")
    return oof

production_base_estimator = clone(selected_artifact["estimator"])
oof_raw = grouped_oof_probabilities(
    production_base_estimator, X_train_pool, y_train_pool, groups_train_pool, n_splits=5
)

OOF fold 1 complete


OOF fold 2 complete


OOF fold 3 complete


OOF fold 4 complete


OOF fold 5 complete


In [3]:
# Sigmoid probability calibration learned only from out-of-fold training predictions.
def safe_logit(p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))

calibrator = LogisticRegression(max_iter=1000, solver="lbfgs")
calibrator.fit(safe_logit(oof_raw).reshape(-1, 1), y_train_pool)

oof_calibrated = calibrator.predict_proba(safe_logit(oof_raw).reshape(-1, 1))[:, 1]
print("OOF Brier score:", round(brier_score_loss(y_train_pool, oof_calibrated), 5))

OOF Brier score: 0.02276


In [4]:
# Tune final operating threshold using OOF predictions only. Test remains untouched.
def tune_threshold(y_true, probability, minimum_recall=0.90):
    rows = []
    for threshold in np.round(np.linspace(0.05, 0.95, 181), 4):
        pred = (probability >= threshold).astype(int)
        rows.append({
            "threshold": threshold,
            "precision": precision_score(y_true, pred, zero_division=0),
            "recall": recall_score(y_true, pred, zero_division=0),
            "f1": f1_score(y_true, pred, zero_division=0)
        })
    table = pd.DataFrame(rows)
    eligible = table[table["recall"] >= minimum_recall]
    chosen = (eligible if len(eligible) else table).sort_values(["f1", "precision"], ascending=False).iloc[0]
    return float(chosen["threshold"]), table

prediction_threshold, threshold_table = tune_threshold(
    y_train_pool.values, oof_calibrated, minimum_recall=0.90
)

print("Production prediction threshold:", prediction_threshold)
threshold_table.to_csv(OUTPUT_DIR / "final_threshold_table.csv", index=False)

Production prediction threshold: 0.5


In [5]:
# Fit the final estimator on ALL non-test groups.
final_estimator = clone(selected_artifact["estimator"])
final_estimator.fit(X_train_pool, y_train_pool)

test_raw_prob = final_estimator.predict_proba(X_test)[:, 1]
test_calibrated_prob = calibrator.predict_proba(safe_logit(test_raw_prob).reshape(-1, 1))[:, 1]
test_pred = (test_calibrated_prob >= prediction_threshold).astype(int)

print(classification_report(
    y_test, test_pred, target_names=["INTENDED", "EXCESSIVE"], zero_division=0
))

test_metrics = {
    "accuracy": accuracy_score(y_test, test_pred),
    "precision_excessive": precision_score(y_test, test_pred, zero_division=0),
    "recall_excessive": recall_score(y_test, test_pred, zero_division=0),
    "f1_excessive": f1_score(y_test, test_pred, zero_division=0),
    "roc_auc": roc_auc_score(y_test, test_calibrated_prob),
    "pr_auc": average_precision_score(y_test, test_calibrated_prob),
    "balanced_accuracy": balanced_accuracy_score(y_test, test_pred),
    "mcc": matthews_corrcoef(y_test, test_pred),
    "brier_score": brier_score_loss(y_test, test_calibrated_prob)
}

print(pd.Series(test_metrics).round(4))

              precision    recall  f1-score   support

    INTENDED       0.99      0.99      0.99      3200
   EXCESSIVE       0.97      0.96      0.96       800

    accuracy                           0.98      4000
   macro avg       0.98      0.98      0.98      4000
weighted avg       0.98      0.98      0.98      4000

accuracy               0.9850
precision_excessive    0.9660
recall_excessive       0.9588
f1_excessive           0.9624
roc_auc                0.9972
pr_auc                 0.9928
balanced_accuracy      0.9752
mcc                    0.9530
brier_score            0.0120
dtype: float64


In [6]:
import sys, platform, json, joblib
from pathlib import Path
import pandas as pd

MODEL_VERSION     = "iam_permission_model_v2_final_20260924"
OPTIMAL_THRESHOLD = 0.535
REMOVE_THRESHOLD  = 0.70
REVIEW_THRESHOLD  = 0.40

PROJECT_DIR  = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
ARTIFACT_DIR = PROJECT_DIR / "artifacts"
DATA_DIR     = PROJECT_DIR / "data"

try:
    _ = candidate
except NameError:
    candidate = joblib.load(ARTIFACT_DIR / "xgboost_candidate.joblib")
try:
    _ = final_estimator
except NameError:
    final_estimator = candidate["estimator"]
try:
    _ = feature_columns
except NameError:
    feature_columns = candidate["feature_columns"]

numeric_features     = candidate["numeric_features"]
categorical_features = candidate["categorical_features"]

raw_catalog_path = DATA_DIR / "raw" / "AWS_Risk_Weight_Table_FINAL_4007_Actions_Shuffled.csv"
risk_catalog = dict(pd.read_csv(raw_catalog_path)[["action", "risk_weight"]].values) if raw_catalog_path.exists() else {}

final_artifact = {
    "model_name":            "xgboost",
    "model_version":         MODEL_VERSION,
    "selection_date":        "2026-09-24",
    "estimator":             final_estimator,
    "model":                 final_estimator,
    "calibrator":            calibrator if 'calibrator' in locals() else None,
    "risk_catalog":          risk_catalog,
    "metadata": {
        "model_version": MODEL_VERSION,
        "prediction_threshold": OPTIMAL_THRESHOLD,
        "feature_columns": feature_columns,
        "numeric_features": numeric_features,
        "categorical_features": categorical_features,
        "test_metrics": test_metrics if 'test_metrics' in locals() else {},
    },
    "feature_columns":       feature_columns,
    "numeric_features":      numeric_features,
    "categorical_features":  categorical_features,
    "optimal_threshold":     OPTIMAL_THRESHOLD,
    "recommendation_thresholds": {
        "REMOVE": REMOVE_THRESHOLD,
        "REVIEW": REVIEW_THRESHOLD,
        "KEEP":   0.0,
    },
    "cv_pr_auc":         candidate["best_cv_pr_auc"],
    "validation_pr_auc": candidate["validation_pr_auc"],
    "best_hyperparams":  candidate["best_params"],
    "policy_notes": [
        "This model produces RECOMMENDATIONS only.",
        "Administrator approval is required before any IAM permission is modified.",
        "The model does not directly modify IAM policies or permissions.",
        "Use inference.py:run_inference() for production output.",
    ],
    "python_version": sys.version,
    "platform":       platform.platform(),
}

final_path = ARTIFACT_DIR / "iam_permission_model_v2_final.joblib"
joblib.dump(final_artifact, final_path)
print(f"Saved : {final_path}")
print(f"Size  : {final_path.stat().st_size / 1024:.1f} KB")
print(f"Keys  : {list(final_artifact.keys())}")


Saved : C:\Users\LENOVO\Downloads\IAM_Model_V2_Notebooks\artifacts\iam_permission_model_v2_final.joblib
Size  : 776.0 KB
Keys  : ['model_name', 'model_version', 'selection_date', 'estimator', 'model', 'calibrator', 'risk_catalog', 'metadata', 'feature_columns', 'numeric_features', 'categorical_features', 'optimal_threshold', 'recommendation_thresholds', 'cv_pr_auc', 'validation_pr_auc', 'best_hyperparams', 'policy_notes', 'python_version', 'platform']


In [7]:
# Production output via inference.py -- guarantees 14-col schema
# ML model produces RECOMMENDATIONS only.
# No IAM permissions are modified. Administrator approval required.
import sys
sys.path.insert(0, str(PROJECT_DIR))
from inference import load_model as _lm, run_inference as _ri

_model  = _lm(ARTIFACT_DIR / "iam_permission_model_v2_final.joblib")
test_df = df.loc[test_mask].reset_index(drop=True).copy()
results = _ri(test_df, _model)

results.to_csv(OUTPUT_DIR / "final_production_predictions.csv", index=False)
results.to_csv(OUTPUT_DIR / "final_predictions.csv", index=False)
print("Saved:", OUTPUT_DIR / "final_production_predictions.csv")
print("Saved:", OUTPUT_DIR / "final_predictions.csv")
print(f"Rows  : {len(results)}")
print(f"Cols  : {list(results.columns)}")
print("Recommendation counts:")
print(results["recommendation"].value_counts())


Saved: C:\Users\LENOVO\Downloads\IAM_Model_V2_Notebooks\outputs\final_production_predictions.csv
Saved: C:\Users\LENOVO\Downloads\IAM_Model_V2_Notebooks\outputs\final_predictions.csv
Rows  : 4000
Cols  : ['recommendation_id', 'user_id', 'role_id', 'action', 'resource', 'risk_score', 'risk_weight', 'risk_level', 'prediction', 'recommendation', 'reason_codes', 'explanation', 'model_version', 'generated_at']
Recommendation counts:
recommendation
KEEP      3147
REMOVE     748
REVIEW     105
Name: count, dtype: int64


In [8]:
import joblib, numpy as np, pandas as pd, sys
from pathlib import Path
from sklearn.metrics import (
    average_precision_score, roc_auc_score, f1_score,
    precision_score, recall_score, brier_score_loss,
)

PROJECT_DIR  = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
ARTIFACT_DIR = PROJECT_DIR / "artifacts"
DATA_DIR     = PROJECT_DIR / "data"
MODEL_VERSION     = "iam_permission_model_v2_final_20260924"
OPTIMAL_THRESHOLD = 0.535

loaded = joblib.load(ARTIFACT_DIR / "iam_permission_model_v2_final.joblib")
assert loaded["model_version"] == MODEL_VERSION
assert "estimator" in loaded
assert "inference_fn" not in loaded
mv  = loaded["model_version"]
thr = loaded["optimal_threshold"]
nf  = len(loaded["feature_columns"])
print(f"[1/3] Artifact reload: PASS")
print(f"      model_version : {mv}")
print(f"      threshold     : {thr}")
print(f"      features      : {nf}")

df     = pd.read_csv(DATA_DIR / "labeled_dataset.csv")
splits = pd.read_csv(DATA_DIR / "splits.csv")
df["split"] = splits["split"].values
test_df = df[df["split"] == "test"].reset_index(drop=True)
y_test  = test_df["target"].astype(int).values
X_test  = test_df[loaded["feature_columns"]]
y_prob  = loaded["estimator"].predict_proba(X_test)[:, 1]
y_pred  = (y_prob >= OPTIMAL_THRESHOLD).astype(int)
print("[2/3] Test-set performance:")
print(f"      PR-AUC    : {average_precision_score(y_test, y_prob):.4f}")
print(f"      ROC-AUC   : {roc_auc_score(y_test, y_prob):.4f}")
print(f"      Precision : {precision_score(y_test, y_pred, zero_division=0):.4f}")
print(f"      Recall    : {recall_score(y_test, y_pred, zero_division=0):.4f}")
print(f"      F1        : {f1_score(y_test, y_pred, zero_division=0):.4f}")
print(f"      Brier     : {brier_score_loss(y_test, y_prob):.4f}")

sys.path.insert(0, str(PROJECT_DIR))
from inference import load_model, run_inference
model   = load_model(ARTIFACT_DIR / "iam_permission_model_v2_final.joblib")
results = run_inference(test_df.head(10), model)
REQUIRED = [
    "recommendation_id", "user_id", "role_id", "action", "resource",
    "risk_score", "risk_weight", "risk_level", "prediction",
    "recommendation", "reason_codes", "explanation", "model_version", "generated_at"
]
assert list(results.columns) == REQUIRED
assert results["recommendation_id"].is_unique
assert results["risk_score"].between(0, 1).all()
nr = len(results); nc = len(results.columns)
print(f"[3/3] inference.py: PASS  ({nr} rows, {nc} cols)")
print("All checks PASSED.")


[1/3] Artifact reload: PASS
      model_version : iam_permission_model_v2_final_20260924
      threshold     : 0.535
      features      : 22
[2/3] Test-set performance:


      PR-AUC    : 0.9928
      ROC-AUC   : 0.9972
      Precision : 0.9566
      Recall    : 0.9637
      F1        : 0.9601
      Brier     : 0.0170


[3/3] inference.py: PASS  (10 rows, 14 cols)
All checks PASSED.
